# Workshop 3 — ETL Streaming con Apache Kafka
## Step 1 & 2: Exploratory Data Analysis (EDA) + Data Cleaning & Harmonization

**Curso:** ETL (G01) — Data Engineering and Artificial Intelligence  
**Dataset:** World Happiness Report 2015–2019

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

---
## 1. Carga de datos

In [ ]:
# Rutas — ajusta si tus CSV están en otra carpeta
DATA_PATH = '../data/raw/'

files = {
    2015: pd.read_csv(DATA_PATH + '2015.csv'),
    2016: pd.read_csv(DATA_PATH + '2016.csv'),
    2017: pd.read_csv(DATA_PATH + '2017.csv'),
    2018: pd.read_csv(DATA_PATH + '2018.csv'),
    2019: pd.read_csv(DATA_PATH + '2019.csv'),
}

for year, df in files.items():
    print(f'\n=== {year} ===')
    print(f'Shape: {df.shape}')
    print(f'Columnas: {list(df.columns)}')

---
## 2. Diferencias de schema entre años

Los datasets **no comparten el mismo schema**. Aquí analizamos qué columnas tiene cada año.

In [ ]:
print('Columnas por año:')
for year, df in files.items():
    print(f'\n{year}:')
    for col in df.columns:
        print(f'  - {col} ({df[col].dtype})')

---
## 3. Análisis de calidad por año

### 3.1 Valores nulos

In [ ]:
print('Valores nulos por columna y año:')
for year, df in files.items():
    nulls = df.isnull().sum()
    if nulls.any():
        print(f'\n{year}:')
        print(nulls[nulls > 0])
    else:
        print(f'\n{year}: Sin valores nulos ✓')

### 3.2 Registros duplicados

In [ ]:
for year, df in files.items():
    dups = df.duplicated().sum()
    print(f'{year}: {dups} duplicados')

### 3.3 Estadísticas descriptivas

In [ ]:
for year, df in files.items():
    print(f'\n=== {year} ===')
    display(df.describe())

---
## 4. Armonización del schema

**Decisión de diseño:** Cada año usa nombres de columna diferentes para los mismos conceptos.
Creamos un mapeo unificado basado en el significado semántico de cada columna.

**Schema unificado propuesto:**

| Campo unificado | Descripción |
|---|---|
| `country` | País |
| `year` | Año del reporte |
| `happiness_score` | Puntaje de felicidad (target) |
| `gdp` | PIB per cápita |
| `family` | Soporte social / familia |
| `health` | Expectativa de vida saludable |
| `freedom` | Libertad para tomar decisiones |
| `generosity` | Generosidad |
| `corruption` | Percepción de corrupción |

In [ ]:
# Mapeo de columnas originales → schema unificado por año
column_maps = {
    2015: {
        'Country': 'country',
        'Happiness Score': 'happiness_score',
        'Economy (GDP per Capita)': 'gdp',
        'Family': 'family',
        'Health (Life Expectancy)': 'health',
        'Freedom': 'freedom',
        'Trust (Government Corruption)': 'corruption',
        'Generosity': 'generosity',
    },
    2016: {
        'Country': 'country',
        'Happiness Score': 'happiness_score',
        'Economy (GDP per Capita)': 'gdp',
        'Family': 'family',
        'Health (Life Expectancy)': 'health',
        'Freedom': 'freedom',
        'Trust (Government Corruption)': 'corruption',
        'Generosity': 'generosity',
    },
    2017: {
        'Country': 'country',
        'Happiness.Score': 'happiness_score',
        'Economy..GDP.per.Capita.': 'gdp',
        'Family': 'family',
        'Health..Life.Expectancy.': 'health',
        'Freedom': 'freedom',
        'Trust..Government.Corruption.': 'corruption',
        'Generosity': 'generosity',
    },
    2018: {
        'Country or region': 'country',
        'Score': 'happiness_score',
        'GDP per capita': 'gdp',
        'Social support': 'family',
        'Healthy life expectancy': 'health',
        'Freedom to make life choices': 'freedom',
        'Perceptions of corruption': 'corruption',
        'Generosity': 'generosity',
    },
    2019: {
        'Country or region': 'country',
        'Score': 'happiness_score',
        'GDP per capita': 'gdp',
        'Social support': 'family',
        'Healthy life expectancy': 'health',
        'Freedom to make life choices': 'freedom',
        'Perceptions of corruption': 'corruption',
        'Generosity': 'generosity',
    },
}

UNIFIED_COLS = ['country', 'year', 'happiness_score', 'gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']

harmonized_dfs = []

for year, df in files.items():
    mapping = column_maps[year]
    df_renamed = df.rename(columns=mapping)
    df_renamed['year'] = year
    # Seleccionar solo columnas del schema unificado
    available = [c for c in UNIFIED_COLS if c in df_renamed.columns]
    df_clean = df_renamed[available].copy()
    harmonized_dfs.append(df_clean)
    print(f'{year}: {df_clean.shape[0]} filas, columnas disponibles: {available}')

df_unified = pd.concat(harmonized_dfs, ignore_index=True)
print(f'\nDataset unificado: {df_unified.shape}')

---
## 5. Limpieza del dataset unificado

In [ ]:
print('Nulos antes de limpieza:')
print(df_unified.isnull().sum())

# Estrategia: imputar nulos numéricos con la mediana por año
numeric_cols = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']

for col in numeric_cols:
    if col in df_unified.columns:
        df_unified[col] = df_unified.groupby('year')[col].transform(
            lambda x: x.fillna(x.median())
        )

# Eliminar filas donde falte el target o el país
df_unified.dropna(subset=['happiness_score', 'country'], inplace=True)

# Normalizar nombre de país
df_unified['country'] = df_unified['country'].str.strip()

print('\nNulos después de limpieza:')
print(df_unified.isnull().sum())
print(f'\nShape final: {df_unified.shape}')

---
## 6. Visualizaciones EDA

In [ ]:
# Distribución del happiness score
plt.figure(figsize=(10, 4))
sns.histplot(df_unified['happiness_score'], bins=30, kde=True, color='steelblue')
plt.title('Distribución del Happiness Score (2015–2019)')
plt.xlabel('Happiness Score')
plt.tight_layout()
plt.show()

In [ ]:
# Correlación entre variables
corr_cols = ['happiness_score', 'gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
corr_df = df_unified[corr_cols].dropna()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: features vs happiness score
features = ['gdp', 'family', 'health', 'freedom']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feat in zip(axes.flatten(), features):
    ax.scatter(df_unified[feat], df_unified['happiness_score'], alpha=0.4, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('happiness_score')
    ax.set_title(f'{feat} vs happiness_score')

plt.suptitle('Relación features vs Happiness Score', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: happiness score por año
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_unified, x='year', y='happiness_score', palette='Set2')
plt.title('Happiness Score por año')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 países más felices (promedio 2015-2019)
top10 = df_unified.groupby('country')['happiness_score'].mean().nlargest(10).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=top10, x='happiness_score', y='country', palette='Blues_r')
plt.title('Top 10 países más felices (promedio 2015–2019)')
plt.xlabel('Happiness Score promedio')
plt.tight_layout()
plt.show()

---
## 7. Detección de outliers

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
cols_to_check = ['happiness_score', 'gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']

for ax, col in zip(axes.flatten(), cols_to_check):
    if col in df_unified.columns:
        sns.boxplot(y=df_unified[col], ax=ax, color='lightcoral')
        ax.set_title(col)

axes.flatten()[-1].set_visible(False)
plt.suptitle('Outliers por variable', fontsize=13)
plt.tight_layout()
plt.show()

# Reporte de outliers usando IQR
print('\nConteo de outliers por variable (método IQR):')
for col in cols_to_check:
    if col in df_unified.columns:
        Q1 = df_unified[col].quantile(0.25)
        Q3 = df_unified[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df_unified[col] < Q1 - 1.5*IQR) | (df_unified[col] > Q3 + 1.5*IQR)).sum()
        print(f'  {col}: {outliers} outliers')

---
## 8. Observaciones de calidad de datos

| Problema | Año(s) afectado(s) | Decisión |
|---|---|---|
| Nombres de columna diferentes | Todos | Mapeo unificado por año |
| Columnas extra (rank, region, etc.) | 2015, 2016, 2017 | Descartadas — no aportan al modelo |
| Valores nulos en `corruption` | 2018 | Imputación con mediana por año |
| Tipos inconsistentes | Algunos años | Forzado a float64 implícito por pandas |
| Outliers leves | Generosity, Corruption | Se conservan — son valores reales del reporte |

---
## 9. Exportar dataset unificado

In [ ]:
output_path = '../data/processed/happiness_unified.csv'
df_unified.to_csv(output_path, index=False)
print(f'Dataset unificado guardado en: {output_path}')
print(f'Shape: {df_unified.shape}')
display(df_unified.head(10))